# 🔥 ClusterOps: Thermal GPU Balancer — GRPO Training

**Train an LLM (Llama-3.1-8B) to manage a GPU data center under adversarial thermal conditions using Group Relative Policy Optimization (GRPO).**

This notebook:
1. Starts the ClusterOps OpenEnv server (via subprocess)
2. Loads a base LLM with 4-bit quantization (Unsloth)
3. Defines reward functions that query the live environment
4. Trains with GRPO (TRL) for curriculum learning across **Operational Scenarios**
5. Plots reward curves and shows before/after agent behavior comparison

**IMPORTANT:** Make sure your GitHub repository is **PUBLIC** so this notebook can clone it.

**Runtime:** Google Colab T4 GPU (free tier works!)

---

## 1. Install Dependencies

In [ ]:
# Install Unsloth for efficient 4-bit training
!pip install unsloth -q
!pip install trl>=0.12.0 transformers>=4.45.0 -q
!pip install fastapi uvicorn requests pydantic openenv-core -q
!pip install matplotlib numpy -q
print('✅ Dependencies installed')

## 2. Clone & Start ClusterOps Environment Server

In [ ]:
import subprocess, time, requests, os

# Clone the repo
!git clone https://github.com/Sushmit-Biswas/thermal-gpu-balancer.git /content/clusterops_repo

os.chdir('/content/clusterops_repo')

# Start the FastAPI server in background
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for server to boot
for i in range(15):
    time.sleep(2)
    try:
        resp = requests.get('http://localhost:8000/health', timeout=3)
        if resp.ok:
            print(f'✅ ClusterOps server online after {(i+1)*2}s')
            break
    except:
        print(f'  Waiting... ({(i+1)*2}s)')
else:
    print('❌ Server failed to start')
    print(server_proc.stderr.read().decode())

In [ ]:
# Verify server and show schema
import json
schema = requests.get('http://localhost:8000/schema').json()
print('Environment Schema:')
print(json.dumps(schema, indent=2))

## 3. Baseline Agent (Before Training)

In [ ]:
ENV_URL = 'http://localhost:8000'

def env_reset(scenario='01_baseline'):
    return requests.post(f'{ENV_URL}/reset', json={'scenario': scenario}).json()

def env_step(action):
    return requests.post(f'{ENV_URL}/step', json=action).json()

def env_rubric():
    return requests.get(f'{ENV_URL}/grader/rubric').json()

def run_heuristic_baseline(scenario='01_baseline', verbose=False):
    """Simple heuristic: allocate highest-priority job to coolest idle node."""
    data = env_reset(scenario)
    obs = data.get('observation', {})
    total_reward = 0.0

    while not data.get('done', False):
        nodes = obs.get('gpu_nodes', [])
        queue = obs.get('job_queue', [])

        # Priority 1: Evict any node above 90°C
        action = {'action_type': 'wait'}
        for n in nodes:
            if n['status'] == 'busy' and n['temperature'] >= 90.0:
                action = {'action_type': 'evict', 'node_id': n['id']}
                break

        # Priority 2: Allocate
        if action['action_type'] == 'wait' and queue:
            idle = sorted([n for n in nodes if n['status'] == 'idle'], key=lambda n: n['temperature'])
            sorted_q = sorted(queue, key=lambda j: {'vip_training': 0, 'inference': 1, 'batch': 2}[j['type']])
            if idle:
                action = {'action_type': 'allocate', 'job_id': sorted_q[0]['id'], 'node_id': idle[0]['id']}

        data = env_step(action)
        obs = data.get('observation', {})
        total_reward += data.get('reward', 0)

    rubric = env_rubric()
    if verbose:
        print(f'  Reward: {total_reward:.1f} | Score: {rubric["total"]:.3f} | '
              f'Safety: {rubric["thermal_safety"]:.2f} | Throughput: {rubric["throughput"]:.2f}')
    return total_reward, rubric

print('Running 5 baseline episodes on 01_baseline scenario...')
baseline_rewards = []
baseline_scores = []
for i in range(5):
    r, rub = run_heuristic_baseline('01_baseline', verbose=True)
    baseline_rewards.append(r)
    baseline_scores.append(rub['total'])

print(f'\n📊 Baseline: avg_reward={sum(baseline_rewards)/5:.1f}, avg_score={sum(baseline_scores)/5:.3f}')

## 4. Load LLM with Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)
print('✅ Model loaded with LoRA adapters')

## 5. GRPO Reward Functions

In [ ]:
import json, re

SYSTEM_PROMPT = """You are an autonomous GPU cluster scheduler. Your goal: schedule jobs to maximize completions and prevent thermal meltdowns.

Actions (respond with ONE JSON object only):
  {"action_type": "allocate", "job_id": "job_1", "node_id": 2}  # assign queued job to idle node
  {"action_type": "evict", "node_id": 5}                         # emergency-stop hot node
  {"action_type": "cooldown", "node_id": 1}                       # force-cool idle node
  {"action_type": "wait"}                                         # do nothing

Prevent meltdowns (temp >= limit = -50 penalty). Complete VIP jobs first (+40 each)."""

def format_obs(obs, metadata):
    nodes = obs.get('gpu_nodes', [])
    queue = obs.get('job_queue', [])
    step = metadata.get('step', '?')
    max_s = metadata.get('max_steps', '?')
    lines = [f'Step {step}/{max_s} | Completed={obs.get("completed_jobs",0)} Meltdowns={obs.get("meltdowns",0)}']
    lines.append('NODES: ' + ' | '.join(
        f'N{n["id"]}:{n["status"][0].upper()}{n["temperature"]:.0f}C'
        + (f'({n.get("job_type","?")[:3]},{n["job_duration_remaining"]})' if n['status'] == 'busy' else '')
        for n in nodes
    ))
    if queue:
        lines.append('QUEUE: ' + ' | '.join(f'{j["id"]}:{j["type"][:3]},d{j["duration"]},w{j["wait_time"]}' for j in queue[:6]))
    else:
        lines.append('QUEUE: empty')
    return '\n'.join(lines)

def parse_action(text):
    try:
        start, end = text.index('{'), text.rindex('}') + 1
        action = json.loads(text[start:end])
        if 'action_type' in action:
            return action
    except:
        pass
    return {'action_type': 'wait'}

@torch.no_grad()
def generate_action(model, tokenizer, prompt_text):
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=80, temperature=0.4, do_sample=True)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def run_llm_episode(model, tokenizer, scenario='01_baseline'):
    data = env_reset(scenario)
    obs = data.get('observation', {})
    meta = data.get('metadata', {})
    total_reward = 0.0

    while not data.get('done', False):
        user_msg = format_obs(obs, meta)
        prompt = (
            f'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}\n'
            f'<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user_msg}\n'
            f'<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n'
        )
        text = generate_action(model, tokenizer, prompt)
        action = parse_action(text)
        data = env_step(action)
        obs = data.get('observation', {})
        meta = data.get('metadata', {})
        total_reward += data.get('reward', 0)

    rubric = env_rubric()
    return total_reward, rubric

print('✅ Reward functions defined')

## 6. GRPO Training Loop with Curriculum Learning

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
os.makedirs('/content/plots', exist_ok=True)

# Curriculum: start with baseline, promote when score >= threshold
CURRICULUM = ['01_baseline', '02_spatial_bleed', '03_heterogeneous']
PROMOTE_THRESHOLD = [0.65, 0.70]  
EPISODES_PER_STAGE = 20  

all_rewards, all_scores, all_scenarios = [], [], []
scenario_idx = 0
window = 5

print('=' * 60)
print('GRPO Curriculum Training: baseline → spatial_bleed → heterogeneous')
print('=' * 60)

for episode in range(EPISODES_PER_STAGE * len(CURRICULUM)):
    scene = CURRICULUM[scenario_idx]
    reward, rubric = run_llm_episode(model, tokenizer, scenario=scene)
    all_rewards.append(reward)
    all_scores.append(rubric['total'])
    all_scenarios.append(scene)

    if (episode + 1) % 5 == 0:
        avg_r = sum(all_rewards[-window:]) / window
        avg_s = sum(all_scores[-window:]) / window
        print(f'  Ep {episode+1:3d} [{scene}] | Avg Reward={avg_r:+7.1f} | '
              f'Score={avg_s:.3f} | Safety={rubric["thermal_safety"]:.2f} | '
              f'Throughput={rubric["throughput"]:.2f}')

    # Curriculum promotion
    if scenario_idx < len(PROMOTE_THRESHOLD):
        recent = all_scores[-window:]
        if len(recent) == window and sum(recent)/window >= PROMOTE_THRESHOLD[scenario_idx]:
            scenario_idx += 1
            print(f'\n  🎓 PROMOTED to {CURRICULUM[scenario_idx]}!\n')

print('\n✅ Training loop complete!')

## 7. Plot Training Curves

In [ ]:
import numpy as np
from matplotlib.patches import Patch

SCENE_COLORS = {'01_baseline': '#4CAF50', '02_spatial_bleed': '#FF9800', '03_heterogeneous': '#F44336'}
episodes = list(range(1, len(all_rewards) + 1))

fig, axes = plt.subplots(2, 1, figsize=(12, 9))

# ── Plot 1: Reward Curve ──
ax = axes[0]
for i, (r, s) in enumerate(zip(all_rewards, all_scenarios)):
    ax.bar(i+1, r, color=SCENE_COLORS.get(s, '#999'), alpha=0.4, width=1.0)

# Rolling average
roll_w = 8
rolling = np.convolve(all_rewards, np.ones(roll_w)/roll_w, mode='valid')
ax.plot(range(roll_w, len(all_rewards)+1), rolling, color='#1A237E', linewidth=2.5, label=f'Rolling avg ({roll_w} ep)')

# Baseline line
bl = sum(baseline_rewards)/len(baseline_rewards)
ax.axhline(bl, color='gray', linestyle='--', linewidth=1.5, label=f'Heuristic baseline ({bl:.0f})')

legend_patches = [Patch(color=c, label=d, alpha=0.6) for d, c in SCENE_COLORS.items()]
legend_patches += ax.get_lines()
ax.legend(handles=legend_patches, fontsize=9)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Total Episode Reward', fontsize=12)
ax.set_title('ClusterOps GRPO Training — Reward Curve', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# ── Plot 2: Composable Rubric Scores ──
ax2 = axes[1]
ax2.plot(episodes, all_scores, color='#7B1FA2', alpha=0.4, linewidth=1, label='Total Score')
roll_s = np.convolve(all_scores, np.ones(roll_w)/roll_w, mode='valid')
ax2.plot(range(roll_w, len(all_scores)+1), roll_s, color='#7B1FA2', linewidth=2.5, label=f'Score rolling avg')

avg_baseline_score = sum(baseline_scores)/len(baseline_scores)
ax2.axhline(avg_baseline_score, color='gray', linestyle='--', linewidth=1.5, label=f'Heuristic score ({avg_baseline_score:.3f})')
ax2.axhline(0.65, color='#4CAF50', linestyle=':', alpha=0.7, label='Promote threshold (baseline→spatial)')
ax2.axhline(0.70, color='#FF9800', linestyle=':', alpha=0.7, label='Promote threshold (spatial→heterogeneous)')
ax2.set_xlabel('Episode', fontsize=12)
ax2.set_ylabel('Rubric Score [0-1]', fontsize=12)
ax2.set_title('Composable Rubric Score', fontsize=11, fontweight='bold')
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/plots/reward_curve.png', dpi=150, bbox_inches='tight')
plt.savefig('/content/clusterops_repo/reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plots saved')

## 8. Before vs. After Comparison

In [ ]:
print('=' * 60)
print('BEFORE vs. AFTER Training Comparison (02_spatial_bleed)')
print('=' * 60)

# Before: run heuristic baseline on spatial
print('\n[BEFORE] Heuristic Baseline (5 episodes, spatial_bleed):')
before_rewards, before_scores = [], []
for i in range(5):
    r, rub = run_heuristic_baseline('02_spatial_bleed', verbose=True)
    before_rewards.append(r)
    before_scores.append(rub['total'])

# After: run LLM agent on spatial
print('\n[AFTER] LLM Agent (5 episodes, spatial_bleed):')
after_rewards, after_scores = [], []
for i in range(5):
    r, rub = run_llm_episode(model, tokenizer, '02_spatial_bleed')
    print(f'  Reward={r:.1f} | Score={rub["total"]:.3f} | Safety={rub["thermal_safety"]:.2f} | Throughput={rub["throughput"]:.2f}')
    after_rewards.append(r)
    after_scores.append(rub['total'])

print(f'\n{'='*60}')
print(f'  Avg Reward: {sum(before_rewards)/5:.1f} → {sum(after_rewards)/5:.1f}  (Δ {(sum(after_rewards)-sum(before_rewards))/5:+.1f})')
print(f'  Avg Score:  {sum(before_scores)/5:.3f} → {sum(after_scores)/5:.3f}  (Δ {(sum(after_scores)-sum(before_scores))/5:+.3f})')
print(f'{'='*60}')

## 9. Save Model

In [ ]:
model.save_pretrained('/content/clusterops_llm')
tokenizer.save_pretrained('/content/clusterops_llm')
print('✅ Model saved to /content/clusterops_llm')
print('   Upload to HuggingFace Hub with: model.push_to_hub("your-hf-username/clusterops-llm")')